# Memory vs Accuracy Tradeoff Demo

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np


class CountMinSketch:
    """Basic Count-Min Sketch implementation for switching activity tracking"""

    def __init__(self, width: int = 1000, depth: int = 5):
        """
        Initialize Count-Min Sketch

        Args:
            width: Number of counters in each hash array
            depth: Number of hash functions (rows)
        """
        self.width = width
        self.depth = depth
        self.sketch = np.zeros((depth, width), dtype=np.float32)

        # Initialize hash function parameters
        self.hash_params = []
        for i in range(depth):
            # Use different random seeds for each hash function
            a = random.randint(1, 2**31)
            b = random.randint(1, 2**31)
            self.hash_params.append((a, b))

    def _hash(self, key: int, row: int) -> int:
        """Hash function for a given key and row"""
        a, b = self.hash_params[row]
        # Universal hash: (a*x + b) mod p mod width
        return ((a * key + b) % 2147483647) % self.width

    def update(self, key: int, value: float = 1.0):
        """
        Update sketch with a value for a key

        Args:
            key: Node ID or hash of node name
            value: Switching activity value to add
        """
        for d in range(self.depth):
            col = self._hash(key, d)
            self.sketch[d][col] += value

    def query(self, key: int) -> float:
        """
        Query estimated value for a key

        Returns:
            Minimum estimate (CMS property: always overestimates)
        """
        estimates = []
        for d in range(self.depth):
            col = self._hash(key, d)
            estimates.append(self.sketch[d][col])

        return min(estimates)

    def merge(self, other: "CountMinSketch") -> "CountMinSketch":
        """Merge another sketch into this one (element-wise addition)"""
        if self.width != other.width or self.depth != other.depth:
            raise ValueError("Sketches must have same dimensions for merging")

        merged = CountMinSketch(self.width, self.depth)
        merged.sketch = self.sketch + other.sketch
        return merged

    def reset(self):
        """Reset all counters to zero"""
        self.sketch.fill(0)


def demo_memory_accuracy_tradeoff():
    """Demonstrate memory vs accuracy trade-off"""
    print("\n" + "=" * 60)
    print("DEMO 5: Memory Usage vs Accuracy Trade-off")
    print("=" * 60)

    # Ground truth data
    np.random.seed(42)

    # Create nodes with different toggle frequencies
    num_nodes = 10000
    print(f"\nGenerating ground truth for {num_nodes:,} nodes...")

    # Power-law distribution (few hot nodes, many cold nodes)
    ground_truth = {}

    # Create hot nodes (top 1%)
    for i in range(num_nodes // 100):
        node_id = i
        # High toggle count (100-500)
        ground_truth[node_id] = np.random.randint(100, 500)

    # Create moderate nodes (next 9%)
    for i in range(num_nodes // 100, num_nodes // 10):
        node_id = i
        # Moderate toggle count (10-100)
        ground_truth[node_id] = np.random.randint(10, 100)

    # Create cold nodes (remaining 90%)
    for i in range(num_nodes // 10, num_nodes):
        node_id = i
        # Low toggle count (0-10)
        ground_truth[node_id] = np.random.randint(0, 10)

    total_toggles = sum(ground_truth.values())
    print(f"Total toggle events: {total_toggles:,}")

    # Test different CMS configurations
    configurations = [
        {"width": 100, "depth": 3, "label": "Tiny (0.3KB)"},
        {"width": 500, "depth": 4, "label": "Small (2KB)"},
        {"width": 2000, "depth": 5, "label": "Medium (40KB)"},
        {"width": 10000, "depth": 6, "label": "Large (240KB)"},
    ]

    results = []

    for config in configurations:
        print(f"\nTesting {config['label']} configuration...")

        # Create CMS
        cms = CountMinSketch(width=config["width"], depth=config["depth"])

        # Update with ground truth
        start_time = time.time()
        for node_id, count in ground_truth.items():
            for _ in range(count):
                cms.update(node_id)
        update_time = time.time() - start_time

        # Query all nodes and calculate error
        errors = []
        relative_errors = []

        query_start = time.time()
        for node_id, true_count in ground_truth.items():
            estimate = cms.query(node_id)
            error = estimate - true_count  # CMS always overestimates
            errors.append(error)

            if true_count > 0:
                relative_errors.append(error / true_count)

        query_time = time.time() - query_start

        # Calculate statistics
        avg_error = np.mean(errors)
        avg_relative_error = np.mean(relative_errors) * 100 if relative_errors else 0
        max_error = np.max(errors)
        error_std = np.std(errors)

        # Calculate memory usage (4 bytes per counter)
        memory_kb = (config["width"] * config["depth"] * 4) / 1024

        # Calculate accuracy for hot nodes (top 100)
        hot_nodes = sorted(ground_truth.items(), key=lambda x: x[1], reverse=True)[:100]
        hot_errors = []

        for node_id, true_count in hot_nodes:
            estimate = cms.query(node_id)
            error = abs(estimate - true_count) / true_count if true_count > 0 else 0
            hot_errors.append(error)

        avg_hot_error = np.mean(hot_errors) * 100 if hot_errors else 0

        result = {
            "config": config["label"],
            "width": config["width"],
            "depth": config["depth"],
            "memory_kb": memory_kb,
            "update_time": update_time,
            "query_time": query_time,
            "avg_error": avg_error,
            "avg_relative_error": avg_relative_error,
            "max_error": max_error,
            "error_std": error_std,
            "avg_hot_error": avg_hot_error,
        }

        results.append(result)

        print(f"  Memory: {memory_kb:6.1f} KB")
        print(f"  Update time: {update_time:.3f}s")
        print(f"  Query time: {query_time:.3f}s")
        print(f"  Avg relative error: {avg_relative_error:.2f}%")
        print(f"  Hot node error: {avg_hot_error:.2f}%")

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Plot 1: Memory vs Accuracy
    config_labels = [r["config"] for r in results]
    memory_values = [r["memory_kb"] for r in results]
    accuracy_values = [100 - r["avg_relative_error"] for r in results]
    hot_accuracies = [100 - r["avg_hot_error"] for r in results]

    axes[0, 0].plot(
        memory_values,
        accuracy_values,
        "bo-",
        linewidth=2,
        markersize=8,
        label="Overall Accuracy",
    )
    axes[0, 0].plot(
        memory_values,
        hot_accuracies,
        "rs--",
        linewidth=2,
        markersize=8,
        label="Hot Node Accuracy",
    )
    axes[0, 0].set_xlabel("Memory Usage (KB)")
    axes[0, 0].set_ylabel("Accuracy (%)")
    axes[0, 0].set_title("Memory vs Accuracy Trade-off")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Add labels for each point
    for i, label in enumerate(config_labels):
        axes[0, 0].annotate(
            label,
            (memory_values[i], accuracy_values[i]),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=9,
        )

    # Plot 2: Update and Query Times
    update_times = [r["update_time"] for r in results]
    query_times = [r["query_time"] for r in results]

    x = range(len(config_labels))
    width = 0.35

    axes[0, 1].bar(
        [i - width / 2 for i in x], update_times, width, label="Update Time", alpha=0.8
    )
    axes[0, 1].bar(
        [i + width / 2 for i in x], query_times, width, label="Query Time", alpha=0.8
    )
    axes[0, 1].set_xlabel("Configuration")
    axes[0, 1].set_ylabel("Time (seconds)")
    axes[0, 1].set_title("Performance Comparison")
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(config_labels, rotation=45)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis="y")

    # Plot 3: Error Distribution for each configuration
    for idx, config in enumerate(configurations[:3]):  # Plot first 3
        # Create histogram of errors for this configuration
        cms = CountMinSketch(width=config["width"], depth=config["depth"])

        # Update with sample data
        sample_nodes = list(ground_truth.keys())[:1000]
        for node_id in sample_nodes:
            count = ground_truth[node_id]
            for _ in range(count):
                cms.update(node_id)

        # Calculate errors
        errors = []
        for node_id in sample_nodes:
            true_count = ground_truth[node_id]
            estimate = cms.query(node_id)
            if true_count > 0:
                rel_error = (estimate - true_count) / true_count * 100
                errors.append(rel_error)

        # Plot histogram
        axes[1, 0].hist(errors, bins=30, alpha=0.5, label=config["label"], density=True)

    axes[1, 0].set_xlabel("Relative Error (%)")
    axes[1, 0].set_ylabel("Density")
    axes[1, 0].set_title("Error Distribution by Configuration")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Error vs Node Activity
    # Use medium configuration for this analysis
    cms = CountMinSketch(width=2000, depth=5)

    # Update all
    for node_id, count in ground_truth.items():
        for _ in range(count):
            cms.update(node_id)

    # Calculate errors vs true count
    node_activities = []
    errors = []

    sample_size = min(500, len(ground_truth))
    sample_indices = np.random.choice(len(ground_truth), sample_size, replace=False)
    sample_keys = list(ground_truth.keys())

    for idx in sample_indices:
        node_id = sample_keys[idx]
        true_count = ground_truth[node_id]
        estimate = cms.query(node_id)

        if true_count > 0:
            node_activities.append(true_count)
            rel_error = (estimate - true_count) / true_count * 100
            errors.append(rel_error)

    scatter = axes[1, 1].scatter(
        node_activities, errors, alpha=0.6, c=node_activities, cmap="viridis", s=20
    )
    axes[1, 1].set_xlabel("True Toggle Count")
    axes[1, 1].set_ylabel("Relative Error (%)")
    axes[1, 1].set_title("Error vs Node Activity (Medium Config)")
    axes[1, 1].set_xscale("log")
    axes[1, 1].grid(True, alpha=0.3)

    # Add colorbar
    plt.colorbar(scatter, ax=axes[1, 1], label="Toggle Count")

    plt.tight_layout()
    plt.savefig("memory_accuracy_tradeoff.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print summary table
    print("\n" + "=" * 80)
    print("SUMMARY TABLE")
    print("=" * 80)
    print(
        "\n{:20s} {:10s} {:10s} {:10s} {:10s} {:10s}".format(
            "Configuration",
            "Memory(KB)",
            "Update(s)",
            "Query(s)",
            "AvgErr(%)",
            "HotErr(%)",
        )
    )
    print("-" * 80)

    for result in results:
        print(
            "{:20s} {:10.1f} {:10.3f} {:10.3f} {:10.2f} {:10.2f}".format(
                result["config"],
                result["memory_kb"],
                result["update_time"],
                result["query_time"],
                result["avg_relative_error"],
                result["avg_hot_error"],
            )
        )

    return results

 Run demo 5

In [ ]:
tradeoff_results = demo_memory_accuracy_tradeoff()

print("\n" + "=" * 60)
print("ALL DEMOS COMPLETED SUCCESSFULLY!")
print("=" * 60)
print("\nSummary of generated files:")
print("  1. switching_power_analysis.png - Power analysis visualization")
print("  2. temporal_current_analysis.png - Temporal and current analysis")
print("  3. pattern_corner_analysis.png - Pattern and corner analysis")
print("  4. memory_accuracy_tradeoff.png - Memory vs accuracy trade-off")
print("\nKey insights demonstrated:")
print("  1. CMS enables efficient switching activity tracking")
print("  2. Memory usage can be 100-1000x less than storing all data")
print("  3. Accuracy is sufficient for identifying hot spots (>95% for hot nodes)")
print("  4. Enables temporal, spatial, and pattern analysis at scale")